# AMML ranking 2026 — финальное решение exp005

Модель: Qwen/Qwen3-Reranker-4B, LoRA. Полный воспроизводящий пайплайн:
зависимости → данные → очистка → фиксированный сплит → токенизация → обучение
одну эпоху → выбор лучшего адаптера по val → инференс → submission.csv.
Измеренный исходный результат: val 0.9538890735, Public 0.946752.
История и анализ экспериментов находятся только в README.md.

Исходная система обучения и инференса: Linux x86_64 (Kaggle), Python 3.13.16,
NVIDIA Tesla T4, CUDA 12.4, PyTorch 2.6.0, fp16. Используется cuda:0.
Локальная проверка структуры и уменьшенного CPU-пайплайна: macOS.
Для повторения полного обучения нужен CUDA GPU; измеренное время — около 10 ч.

В чистом окружении сначала установить requirements.txt и выбрать его Python kernel.
На Kaggle добавить данные соревнования, включить GPU T4 и Internet, затем
Save Version → Save & Run All. Код не загружает готовый обученный адаптер:
он дообучает исходные предобученные веса на train соревнования.


## 1. Проверка и установка зафиксированных зависимостей


In [ ]:
import importlib.metadata as im
import subprocess
import sys
from pathlib import Path
from packaging.requirements import Requirement

EMBEDDED_REQUIREMENTS = 'accelerate==1.6.0\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncertifi==2026.11.20\ncharset-normalizer==3.5.2\ncomm==0.2.3\ndebugpy==1.8.22\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfilelock==4.2.12\nfsspec==2026.9.0\nhuggingface-hub==0.30.2\nidna==3.20\nipykernel==6.29.5\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\nJinja2==3.1.6\njsonschema-specifications==2025.9.1\njsonschema==4.26.0\njupyter_client==8.10.0\njupyter_core==5.9.1\nMarkupSafe==3.0.4\nmatplotlib-inline==0.2.2\nmpmath==1.3.0\nnbclient==0.10.2\nnbformat==5.10.4\nnest-asyncio==1.6.0\nnetworkx==3.7\nnumpy==2.1.3\nnvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-cuda-runtime-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-cufft-cu12==11.2.1.3; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-curand-cu12==10.3.5.147; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-cusolver-cu12==11.6.1.9; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-cusparse-cu12==12.3.1.170; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-cusparselt-cu12==0.6.2; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-nccl-cu12==2.21.5; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-nvjitlink-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64"\nnvidia-nvtx-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64"\npackaging==26.3\npandas==2.3.3\nparso==0.8.7\npeft==0.15.2\npexpect==4.9.0; sys_platform != "win32"\nplatformdirs==4.14.8\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npython-dateutil==4.0.0\npytz==2026.6\nPyYAML==6.0.4\npyzmq==27.3.0\nreferencing==0.37.0\nregex==2026.11.19\nrequests==2.34.2\nrpds-py==2026.9.1\nsafetensors==0.5.3\nsetuptools==84.2.1\nstack-data==0.6.3\nsympy==1.13.1\ntokenizers==0.21.1\ntorch==2.6.0\ntornado==6.6.1\ntqdm==4.67.1\ntraitlets==5.16.1\ntransformers==4.51.3\ntriton==3.2.0; platform_system == "Linux" and platform_machine == "x86_64"\ntyping_extensions==4.16.0\ntzdata==2026.6\nurllib3==2.8.0\nwcwidth==0.9.4\n'
requirements_path=Path('requirements.txt')
lock_text=requirements_path.read_text() if requirements_path.is_file() else EMBEDDED_REQUIREMENTS
# The notebook can be imported into Kaggle on its own; the fallback is identical
# to the published requirements.txt. Linux CUDA wheels are platform-conditional.
if lock_text.strip()!=EMBEDDED_REQUIREMENTS.strip():
    raise RuntimeError('requirements.txt differs from this notebook; use the matching repository files.')
active=[]
for line in lock_text.splitlines():
    line=line.strip()
    if not line or line.startswith('#'):continue
    requirement=Requirement(line)
    if requirement.marker is None or requirement.marker.evaluate():active.append(requirement)
def installed_version(name):
    try:return im.version(name)
    except im.PackageNotFoundError:return None
need=[r for r in active if installed_version(r.name) is None or
      installed_version(r.name) not in r.specifier]
if need:
    print('Installing pinned dependencies:',len(need),flush=True)
    subprocess.check_call([sys.executable,'-m','pip','install','--no-input']+[str(r) for r in active])
# The text-only model does not use torchvision. Kaggle images can contain an
# incompatible optional torchvision build; remove it before model imports.
if installed_version('torchvision') is not None:
    subprocess.check_call([sys.executable,'-m','pip','uninstall','-y','torchvision'])
module_names={'huggingface-hub':'huggingface_hub'}
stale=[]
core={'torch','transformers','peft','accelerate','numpy','pandas','tokenizers','huggingface-hub','safetensors','tqdm'}
for r in active:
    if r.name not in core:continue
    module=sys.modules.get(module_names.get(r.name,r.name.replace('-','_')))
    loaded=getattr(module,'__version__',None) if module is not None else None
    if loaded is not None and loaded.split('+')[0] not in r.specifier:
        stale.append(f'{r.name}: loaded {loaded}, required {r.specifier}')
if stale:
    raise RuntimeError('Restart the Python kernel after installing requirements, then Run All. '+ '; '.join(stale))
import numpy as np
import pandas as pd
import torch
import transformers
print('Python:',sys.version.split()[0],'| torch:',torch.__version__,'| transformers:',transformers.__version__)
print('numpy:',np.__version__,'| pandas:',pd.__version__,'| CUDA:',torch.cuda.is_available())


## 2. Конфигурация, среда и воспроизводимость

Один запуск = одна папка `run_dir` с артефактами. Если `submission.csv` в ней уже лежит, код требует новый `run_name`, чтобы не затирать прошлый эксперимент.

In [ ]:
import gc
import hashlib
import html as html_lib
import json
import math
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import random
import re
import subprocess
import sys
import time
import unicodedata

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import get_linear_schedule_with_warmup

CFG = {
    'run_name': 'exp005_qwen3_rer4b_lora_seed42_recovery',
    'data_dir': '',            # пусто = автопоиск; иначе путь к папке с train.csv/test.csv
    'out_root': '/kaggle/working',
    'model_id': 'Qwen/Qwen3-Reranker-4B',
    'model_revision': '22e683669bc0f0bd69640a1354a6d0aebcfeede5',
    'seed': 42,                # сплит — фиксирован 42 у всех экспериментов (сравнимость val)
    'train_seed': 42,
    'max_len': 512,            # оверхед chat-шаблона (~80) + query_budget + doc_budget
    'query_budget': 96,
    'doc_budget': 284,
    'doc_char_limit': 2500,    # предобрезка doc-текста по символам (ускоряет токенизацию)
    'batch_size': 4,           # 4B в fp16: VRAM T4 ограничена
    'grad_accum': 8,           # эффективный train batch = 32 (как у остальных)
    'eval_batch_size': 16,
    'epochs': 1,               # пик у всех наших прогонов был в 1-й эпохе; бюджет сессии 12 ч
    'lr': 1e-4,                # LoRA: на 1-2 порядка выше полного файнтюна
    'weight_decay': 0.01,
    'warmup_ratio': 0.10,
    'max_grad_norm': 1.0,
    'eval_every_steps': 200,   # валидация каждые N optimizer-шагов
    'val_fraction': 0.2,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
}

if not (os.path.isdir(CFG['out_root']) and os.access(CFG['out_root'], os.W_OK)):
    CFG['out_root'] = os.getcwd()
    print('"/kaggle/working" недоступен — out_root =', CFG['out_root'])

run_dir = os.path.join(CFG['out_root'], CFG['run_name'])
if os.path.exists(os.path.join(run_dir, 'submission.csv')):
    raise RuntimeError(f'В {run_dir} уже есть submission.csv — задай новый CFG["run_name"].')
os.makedirs(run_dir, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    cap = torch.cuda.get_device_capability(0)
    amp_dtype = torch.bfloat16 if cap >= (8, 0) else torch.float16
    props = torch.cuda.get_device_properties(0)
    print('GPU      :', torch.cuda.get_device_name(0), '| capability', cap,
          '| VRAM ГБ:', round(props.total_memory / 2**30, 1))
else:
    amp_dtype = torch.float32
    print('CUDA недоступна — обучение пойдёт на CPU (очень медленно).')
print('run_dir  :', run_dir)
print('amp dtype:', amp_dtype)
assert device.type == "cuda", "Full 4B training requires a CUDA GPU; see README.md."


## 3. Загрузка данных

Автопоиск папки с `train.csv`/`test.csv`, контроль схемы, уникальности `id` и SHA-256 исходных файлов (фиксируются в артефактах запуска).

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def find_data_dir(cfg):
    candidates = []
    if cfg['data_dir']:
        candidates.append(cfg['data_dir'])
    candidates.append('.')
    root = '/kaggle/input'
    if os.path.isdir(root):
        for dirpath, _, filenames in os.walk(root):
            if 'train.csv' in filenames and 'test.csv' in filenames:
                candidates.append(dirpath)
    for c in candidates:
        if os.path.isfile(os.path.join(c, 'train.csv')) and os.path.isfile(os.path.join(c, 'test.csv')):
            return c
    raise FileNotFoundError('Не найдена папка с train.csv и test.csv — заполни CFG["data_dir"].')

DATA_DIR = find_data_dir(CFG)
train_csv_path = os.path.join(DATA_DIR, 'train.csv')
test_csv_path = os.path.join(DATA_DIR, 'test.csv')
print('DATA_DIR =', DATA_DIR)

train_hash = sha256_file(train_csv_path)
test_hash = sha256_file(test_csv_path)
print('train.csv sha256:', train_hash)
print('test.csv  sha256:', test_hash)

REQUIRED = ['id', 'query_id', 'document_id', 'article_id', 'query', 'title', 'text',
            'passage_index', 'article_passage_count', 'article_char_count', 'ingest_source']

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

assert set(REQUIRED).issubset(train_df.columns), 'train.csv: не хватает колонок'
assert set(REQUIRED).issubset(test_df.columns), 'test.csv: не хватает колонок'
assert not train_df['id'].duplicated().any(), 'train.csv: дубликаты id'
assert not test_df['id'].duplicated().any(), 'test.csv: дубликаты id'


EXPECTED_TRAIN_SHA256='2c1b26c5d3ee642f2c959b758cea49d4176ca4c0dfe92fbb7236733eff801042'
EXPECTED_TEST_SHA256='4667c49296b72bf64767338c5345cb2f6c7cfbd0671364e5b728d9d5bcc2bf20'
assert train_hash==EXPECTED_TRAIN_SHA256,'Unexpected train.csv'
assert test_hash==EXPECTED_TEST_SHA256,'Unexpected test.csv'
assert set(train_df['query_id']).isdisjoint(set(test_df['query_id']))
assert train_df['relevance'].isin([0,1]).all()
print('Input files verified:',len(train_df),'train rows,',len(test_df),'test rows')


## 4. Очистка текстов

Тексты содержат HTML-теги и entities, NBSP и неразрывные символы. Очистка: NFC-нормализация → удаление HTML-комментариев → `html.unescape` (до фикспойнта) → удаление известных тегов → схлопывание пробелов. Регистр, пунктуация и математические записи вида `x < 5` сохраняются: шаблон тега требует известного имени тега. Очищаются только `query`, `title`, `text`; метаданные и id не трогаются.

In [ ]:
COMMENT_RE = re.compile(r'<!--.*?-->', re.DOTALL)
TAG_NAME = (
    r'p|br|hr|div|span|a|b|strong|i|em|u|s|small|sub|sup|code|pre|blockquote|'
    r'h[1-6]|ul|ol|li|dl|dt|dd|table|thead|tbody|tfoot|tr|td|th|caption|'
    r'figure|figcaption|img|center|font|section|article|header|footer|nav|aside|abbr|cite|mark'
)
TAG_RE = re.compile(r'</?(?:' + TAG_NAME + r')(?:\s[^<>]*)?/?>', re.IGNORECASE)
WS_RE = re.compile(r'\s+')

def clean_text(s):
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFC', s)
    s = s.replace('\u00a0', ' ').replace('\u00ad', '').replace('\u200b', '')
    s = COMMENT_RE.sub(' ', s)
    prev = None
    for _ in range(3):                      # разворачиваем цепочки &amp;amp; до фикспойнта
        s = html_lib.unescape(s)
        if s == prev:
            break
        prev = s
    s = TAG_RE.sub(' ', s)
    s = WS_RE.sub(' ', s).strip()
    return s

for df in (train_df, test_df):
    for col in ('query', 'title', 'text'):
        df[col + '_c'] = df[col].map(clean_text)



## 5. NDCG@10 для выбора чекпойнта


In [ ]:
def ndcg_at_10(df, scores):
    """Средний NDCG@10 по запросам; df — колонки id, query_id, relevance."""
    ids = df['id'].astype(str).to_numpy()
    qids = df['query_id'].to_numpy()
    rel = df['relevance'].astype(float).to_numpy()
    scores = np.asarray(scores, dtype=np.float64)
    # primary: score по убыванию; secondary: id по возрастанию как строке
    order = np.lexsort((ids, -scores))
    tmp = pd.DataFrame({'q': qids[order], 'r': rel[order], 'i': ids[order]})
    total, counted = 0.0, 0
    for _, g in tmp.groupby('q', sort=False):
        rels = g['r'].to_numpy()[:10]
        disc = 1.0 / np.log2(np.arange(len(rels)) + 2.0)
        dcg = float(((2.0 ** rels) - 1.0) @ disc)
        ideal = np.sort(g['r'].to_numpy())[::-1][:10]
        idcg = float(((2.0 ** ideal) - 1.0) @ (1.0 / np.log2(np.arange(len(ideal)) + 2.0)))
        if idcg > 0:
            total += dcg / idcg
            counted += 1
    return total / counted if counted else 0.0



## 6. Фиксированное разделение по запросам


In [ ]:
def query_hash(qid, seed):
    return hashlib.sha256(f'{seed}:{qid}'.encode('utf-8')).hexdigest()

all_qids = np.array(sorted(train_df['query_id'].unique()))
hexvals = np.array([int(query_hash(q, CFG['seed'])[:8], 16) for q in all_qids], dtype=np.float64)
is_val = (hexvals / float(2**32)) < CFG['val_fraction']
qid2part = dict(zip(all_qids, np.where(is_val, 'val', 'fit')))

train_df['part'] = train_df['query_id'].map(qid2part)
fit_df = train_df[train_df['part'] == 'fit'].copy().reset_index(drop=True)
val_df = train_df[train_df['part'] == 'val'].copy().reset_index(drop=True)

assert set(fit_df['query_id']).isdisjoint(set(val_df['query_id']))
assert len(fit_df) + len(val_df) == len(train_df)

split_out = pd.DataFrame({'query_id': all_qids, 'part': np.where(is_val, 'val', 'fit')})
split_out.to_csv(os.path.join(run_dir, 'split.csv'), index=False)


print('fit:',len(fit_df),'rows; val:',len(val_df),'rows')


## 7. Токенизация: chat-шаблон Qwen3-Reranker

Пара рендерится шаблоном модели: system-блок судьи → `<Instruct>/<Query>/<Document>` →
assistant-блок `<think></think>`; скор читается с последней позиции (следующий токен —
yes/no). Бюджеты по частям: query ≤ 96 токенов, doc ≤ 284 (текст → декод обратно в
строку для шаблона), предобрезка doc 2500 символов; вместе с оверхедом шаблона всё
гарантированно ≤ max_len 512. Паддинг — слева, чтобы последняя позиция была настоящей.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG['model_id'], revision=CFG['model_revision'])
tokenizer.padding_side = 'left'   # скор на последней позиции — паддинг слева
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
PAD = tokenizer.pad_token_id
YES_ID = tokenizer.convert_tokens_to_ids('yes')
NO_ID = tokenizer.convert_tokens_to_ids('no')
assert YES_ID is not None and NO_ID is not None, 'в словаре токенизатора нет yes/no'
print('yes/no/pad ids:', YES_ID, NO_ID, PAD)

RERANKER_INSTRUCTION = 'Given a web search query, retrieve relevant passages that answer the query'

def build_doc_text(title_c, text_c):
    t = (title_c or '').strip()
    x = (text_c or '').strip()
    if t and x:
        sep = ' ' if t.endswith(('.', '!', '?')) else '. '
        return t + sep + x
    return t or x

def encode_pair(q, doc):
    # бюджет по частям: токенизуем каждую часть, обрезаем до лимита, декодируем
    # обратно в строку для chat-шаблона — суммарная длина гарантирована даже
    # с оверхедом шаблона
    q_ids = tokenizer(q, add_special_tokens=False, truncation=True,
                      max_length=CFG['query_budget'])['input_ids']
    d_ids = tokenizer(doc, add_special_tokens=False, truncation=True,
                      max_length=CFG['doc_budget'])['input_ids']
    messages = [
        {'role': 'system', 'content': RERANKER_INSTRUCTION},
        {'role': 'query', 'content': tokenizer.decode(q_ids)},
        {'role': 'document', 'content': tokenizer.decode(d_ids)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)

def encode_frame(df, desc):
    docs = [build_doc_text(t, x)[:CFG['doc_char_limit']]
            for t, x in zip(df['title_c'], df['text_c'])]
    queries = df['query_c'].tolist()
    out = []
    for q, d in tqdm(list(zip(queries, docs)), total=len(queries), desc=desc):
        out.append(encode_pair(q, d))
    return out

X_fit_ids = encode_frame(fit_df, 'tokenize fit')
X_val_ids = encode_frame(val_df, 'tokenize val')
X_test_ids = encode_frame(test_df, 'tokenize test')

assert max(len(s) for s in X_fit_ids + X_val_ids + X_test_ids) <= CFG['max_len']


## 8. Dataset и DataLoader

Динамический padding по батчу (pad = `<|endoftext|>`), паддинг СЛЕВА — последняя позиция всегда настоящая (там читается скор). Для точной нормировки loss на неполном последнем окне накопления заранее считается план окон (число микробатчей и суммарный размер окна на каждый optimizer-шаг).

In [ ]:
class PairDataset(Dataset):
    def __init__(self, ids_list, labels=None):
        self.ids_list = ids_list
        self.labels = labels

    def __len__(self):
        return len(self.ids_list)

    def __getitem__(self, i):
        if self.labels is None:
            return {'ids': self.ids_list[i]}
        return {'ids': self.ids_list[i], 'label': float(self.labels[i])}

def collate(batch):
    maxlen = max(len(b['ids']) for b in batch)
    ids = torch.full((len(batch), maxlen), PAD, dtype=torch.long)
    att = torch.zeros((len(batch), maxlen), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b['ids'])
        ids[i, maxlen - n:] = torch.tensor(b['ids'], dtype=torch.long)  # паддинг слева
        att[i, maxlen - n:] = 1
    out = {'input_ids': ids, 'attention_mask': att}
    if 'label' in batch[0]:
        out['labels'] = torch.tensor([b['label'] for b in batch], dtype=torch.float32)
    return out

def move_to_device(b):
    return {k: v.to(device, non_blocking=True) for k, v in b.items()}

pin = device.type == 'cuda'
train_loader = DataLoader(PairDataset(X_fit_ids, fit_df['relevance'].to_numpy()),
                          batch_size=CFG['batch_size'], shuffle=True,
                          collate_fn=collate, num_workers=2, pin_memory=pin)

def window_plan(n_items, batch_size, accum):
    """План optimizer-шагов: (число микробатчей в окне, суммарный размер окна)."""
    sizes = [min(batch_size, n_items - s) for s in range(0, n_items, batch_size)]
    micros, samples, cur = [], [], []
    for s in sizes:
        cur.append(s)
        if len(cur) == accum:
            micros.append(len(cur))
            samples.append(sum(cur))
            cur = []
    if cur:
        micros.append(len(cur))
        samples.append(sum(cur))
    return micros, samples

micros_plan, samples_plan = window_plan(len(X_fit_ids), CFG['batch_size'], CFG['grad_accum'])
print(f'микробатчей/эпоху: {len(X_fit_ids) // CFG["batch_size"] + (len(X_fit_ids) % CFG["batch_size"] > 0)}, '
      f'optimizer-шагов/эпоху: {len(samples_plan)}, окон неполных: {sum(1 for m in micros_plan if m < CFG["grad_accum"])}')

## 9. Обучение LoRA с сохранением лучшего адаптера


In [ ]:
from peft import LoraConfig, get_peft_model
model = AutoModelForCausalLM.from_pretrained(CFG['model_id'], revision=CFG['model_revision'], torch_dtype=torch.float16)
lora_cfg = LoraConfig(r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'], lora_dropout=CFG['lora_dropout'], target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], task_type='CAUSAL_LM')
model = get_peft_model(model, lora_cfg)
model.config.use_cache = False
model.enable_input_require_grads()
model.to(device)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.print_trainable_parameters()
no_decay = ('bias', 'LayerNorm.weight')
decay_params = [p for n, p in model.named_parameters() if p.requires_grad and (not any((nd in n for nd in no_decay)))]
nodecay_params = [p for n, p in model.named_parameters() if p.requires_grad and any((nd in n for nd in no_decay))]
optimizer = torch.optim.AdamW([{'params': decay_params, 'weight_decay': CFG['weight_decay']}, {'params': nodecay_params, 'weight_decay': 0.0}], lr=CFG['lr'])
steps_per_epoch = len(samples_plan)
total_steps = steps_per_epoch * CFG['epochs']
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * CFG['warmup_ratio']), num_training_steps=total_steps)
scaler = torch.amp.GradScaler('cuda', enabled=amp_dtype == torch.float16)
print(f"optimizer-шагов всего: {total_steps}, warmup: {int(total_steps * CFG['warmup_ratio'])}")

@torch.no_grad()
def predict_ids(ids_list, batch_size):
    """Логиты для списка токенизованных пар; порядок вывода = порядок входа."""
    model.eval()
    loader = DataLoader(PairDataset(ids_list), batch_size=batch_size, shuffle=False, collate_fn=collate, num_workers=0, pin_memory=pin)
    outs = []
    for batch in loader:
        batch = move_to_device(batch)
        with torch.autocast('cuda', dtype=amp_dtype, enabled=amp_dtype != torch.float32):
            logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits[:, -1, :]
        outs.append((logits[:, YES_ID] - logits[:, NO_ID]).float().cpu().numpy())
    return np.concatenate(outs)

def eval_val():
    scores = predict_ids(X_val_ids, CFG['eval_batch_size'])
    return (float(ndcg_at_10(val_df, scores)), scores)
log_path = os.path.join(run_dir, 'training_log.jsonl')
log_f = open(log_path, 'a', encoding='utf-8')

def log_line(record):
    log_f.write(json.dumps(record, ensure_ascii=False) + '\n')
    log_f.flush()
best = {'ndcg': -1.0, 'step': -1}
best_model_dir = os.path.join(run_dir, 'best_model')

def save_best(ndcg, step, scores):
    import shutil
    import uuid
    candidate = os.path.join(run_dir, 'checkpoint_step_' + str(step) + '_' + uuid.uuid4().hex[:8])
    model.save_pretrained(candidate, safe_serialization=True)
    tokenizer.save_pretrained(candidate)
    predictions = val_df.copy()
    predictions['prediction'] = scores
    predictions.to_csv(os.path.join(candidate, 'validation_predictions.csv'), index=False)
    record = {'ndcg': float(ndcg), 'step': int(step)}
    with open(os.path.join(candidate, 'checkpoint_metrics.json'), 'w') as f:
        json.dump(record, f, indent=2)
    previous = best_model_dir + '_previous'
    if os.path.isdir(previous):
        shutil.rmtree(previous)
    if os.path.isdir(best_model_dir):
        os.replace(best_model_dir, previous)
    os.replace(candidate, best_model_dir)
    best.update(record)
    if os.path.isdir(previous):
        shutil.rmtree(previous)
    print(f'CHECKPOINT SAVED: step {step}, val {ndcg:.4f} -> {best_model_dir}', flush=True)
t_start = time.time()
global_step = 0
for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    ep_micros, ep_samples = window_plan(len(X_fit_ids), CFG['batch_size'], CFG['grad_accum'])
    run_loss, run_n = (0.0, 0)
    wi, micro_cnt = (0, 0)
    for batch in train_loader:
        batch = move_to_device(batch)
        with torch.autocast('cuda', dtype=amp_dtype, enabled=amp_dtype != torch.float32):
            logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
            two = logits[:, -1, :][:, [NO_ID, YES_ID]].float()
            loss_sum = F.cross_entropy(two, batch['labels'].long(), reduction='sum')
        scaler.scale(loss_sum / ep_samples[wi]).backward()
        run_loss += float(loss_sum.item())
        run_n += int(batch['labels'].size(0))
        micro_cnt += 1
        if micro_cnt == ep_micros[wi]:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            log_line({'event': 'train', 'step': global_step, 'epoch': epoch, 'loss': run_loss / max(run_n, 1), 'lr': scheduler.get_last_lr()[0], 'elapsed_min': round((time.time() - t_start) / 60, 1)})
            if global_step % 20 == 0:
                print(f'train step {global_step}/{total_steps} | {(time.time() - t_start) / 60:.1f} мин', flush=True)
            run_loss, run_n = (0.0, 0)
            wi += 1
            micro_cnt = 0
            if global_step == 60 or global_step % CFG['eval_every_steps'] == 0:
                ndcg, checkpoint_scores = eval_val()
                log_line({'event': 'eval', 'step': global_step, 'val_ndcg_at_10': ndcg})
                print(f'step {global_step:5d} | val NDCG@10 = {ndcg:.4f} | {(time.time() - t_start) / 60:.0f} мин')
                if ndcg > best['ndcg']:
                    save_best(ndcg, global_step, checkpoint_scores)
            model.train()
    if global_step % CFG['eval_every_steps'] != 0:
        ndcg, checkpoint_scores = eval_val()
        log_line({'event': 'eval', 'step': global_step, 'val_ndcg_at_10': ndcg})
        print(f'эпоха {epoch} завершена | step {global_step} | val NDCG@10 = {ndcg:.4f} | {(time.time() - t_start) / 60:.0f} мин')
        if ndcg > best['ndcg']:
            save_best(ndcg, global_step, checkpoint_scores)
    model.train()
assert best['step'] > 0 and os.path.isdir(best_model_dir), 'Нет сохранённого checkpoint'
log_f.close()
gc.collect()
best_step_global = best['step']
print(f"Лучший чекпойнт: step {best_step_global}, val NDCG@10 = {best['ndcg']:.4f} → {best_model_dir}")


## 10. Загрузка лучшего адаптера и итоговая валидация


In [ ]:
# Optimizer parameter groups also hold the model tensors alive.
optimizer.zero_grad(set_to_none=True)
del optimizer, scheduler, scaler, decay_params, nodecay_params, logits, two, loss_sum
# Input-grad hooks/checkpoint closures are released along with the model.
del model
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

from peft import PeftModel

best_model_dir = os.path.join(run_dir, 'best_model')
base = AutoModelForCausalLM.from_pretrained(CFG['model_id'], revision=CFG['model_revision'],
                                            torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, best_model_dir)
model.config.use_cache = False
model.to(device)

val_scores = predict_ids(X_val_ids, CFG['eval_batch_size'])
val_ndcg = float(ndcg_at_10(val_df, val_scores))
const_ndcg = float(ndcg_at_10(val_df, np.zeros(len(val_df))))
print(f'val NDCG@10 (лучшая модель): {val_ndcg:.4f}')
print(f'val NDCG@10 (константный бейзлайн): {const_ndcg:.4f}')

val_pred = val_df[['id', 'query_id', 'document_id', 'article_id', 'ingest_source',
                   'passage_index', 'relevance']].copy()
val_pred['prediction'] = val_scores
val_pred.to_csv(os.path.join(run_dir, 'validation_predictions.csv'), index=False)

metrics = {
    'run_name': CFG['run_name'],
    'best_step': best_step_global,
    'val_ndcg_at_10': val_ndcg,
    'val_constant_ndcg_at_10': const_ndcg,
    'val_rows': int(len(val_df)),
    'val_queries': int(val_df['query_id'].nunique()),
    'val_positives': int(val_df['relevance'].sum()),
}
with open(os.path.join(run_dir, 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

## 11. Инференс на test и submission.csv

Все 13 507 тестовых пар получают скор. Проверки: конечность скоров, соответствие множеству `id` теста и `sample_submission.csv` (если найден), порядок строк выравнивается по сэмплу. `submission.csv` пишется в корень output и в папку запуска.

In [ ]:
model.eval()
test_scores = predict_ids(X_test_ids, CFG['eval_batch_size'])
assert np.isfinite(test_scores).all(), 'нефининые скоры на test'
assert len(test_scores) == len(test_df)

sub = pd.DataFrame({'id': test_df['id'], 'prediction': test_scores})

sample_path = None
for name in ('sample_submission.csv',):
    p = os.path.join(DATA_DIR, name)
    if os.path.isfile(p):
        sample_path = p
if sample_path:
    sample = pd.read_csv(sample_path)
    assert list(sample.columns) == ['id', 'prediction'], 'неожиданные колонки sample_submission'
    assert set(sample['id']) == set(sub['id']), 'id сэмпла и предсказаний не совпадают'
    sub = sub.set_index('id').loc[sample['id']].reset_index()
    print('sample_submission найден — порядок строк выровнен по нему.')

assert sub['prediction'].notna().all()
sub_path = os.path.join(run_dir, 'submission.csv')
sub.to_csv(sub_path, index=False)
sub.to_csv(os.path.join(CFG['out_root'], 'submission.csv'), index=False)
print('submission.csv:', sub_path)
print(sub.head(3).to_string(index=False))
print('описание prediction:', sub['prediction'].describe().to_dict())

## 12. Метаданные системы и артефакты воспроизведения


In [ ]:
def pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return None

import importlib.metadata
import platform

env = {
    'run_name': CFG['run_name'],
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'machine': platform.machine(),
    'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
    'gpu_capability': list(torch.cuda.get_device_capability(0)) if device.type == 'cuda' else None,
    'cuda_runtime': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version() if device.type == 'cuda' else None,
    'torch': pkg_version('torch'),
    'accelerate': pkg_version('accelerate'),
    'tqdm': pkg_version('tqdm'),
    'transformers': pkg_version('transformers'),
    'peft': pkg_version('peft'),
    'tokenizers': pkg_version('tokenizers'),
    'pandas': pkg_version('pandas'),
    'numpy': pkg_version('numpy'),
    'amp_dtype': str(amp_dtype),
    'model_id': CFG['model_id'],
    'model_revision': CFG['model_revision'],
    'seed': CFG['seed'],
    'train_seed': CFG.get('train_seed', CFG['seed']),
    'lora_r': CFG.get('lora_r'),
    'max_len': CFG['max_len'],
    'files': {'train.csv': train_hash, 'test.csv': test_hash},
    'finished_at': time.strftime('%Y-%m-%d %H:%M:%S'),
}
with open(os.path.join(run_dir, 'environment.json'), 'w', encoding='utf-8') as f:
    json.dump(env, f, ensure_ascii=False, indent=2)

elapsed_min = (time.time() - t_start) / 60
summary = f"""Запуск: {CFG['run_name']}
Модель: {CFG['model_id']} @ {CFG['model_revision']}
Вход: query -> title+text, max_len={CFG['max_len']} (query<={CFG['query_budget']}, doc<={CFG['doc_budget']}), очистка HTML/entities/NBSP
Split: 80/20 по query_id, seed={CFG['seed']} (SHA-256), fit={len(fit_df)} строк / {fit_df['query_id'].nunique()} запросов, val={len(val_df)} строк / {val_df['query_id'].nunique()} запросов
Обучение: LoRA CE (yes/no), r={CFG.get('lora_r')}, lr={CFG['lr']}, effective batch={CFG['batch_size'] * CFG['grad_accum']}, epochs={CFG['epochs']}, precision={amp_dtype}
Лучший чекпойнт: step {best_step_global}, val NDCG@10 = {val_ndcg:.4f}
Константный бейзлайн на val: {const_ndcg:.4f}
Время: {elapsed_min:.0f} мин
ОС/GPU: {env['platform']} / {env['gpu']}
Артефакты: {run_dir}
Submission: {os.path.join(CFG['out_root'], 'submission.csv')}
"""
with open(os.path.join(run_dir, 'run_summary.txt'), 'w', encoding='utf-8') as f:
    f.write(summary)
log_f.close()
print(summary)
# Record the complete installed dependency set alongside the run artifacts.
with open(os.path.join(run_dir,'installed_packages.txt'),'w',encoding='utf-8') as f:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)
